# Full study: from a score to a kernel

This lesson uses only numbers already proved. The point is to see them as one machine.

Training points:

$$
(2, 0),\ y = +1
\qquad
(0, 1),\ y = -1
\qquad
(0, -1),\ y = -1
$$

Hard-margin solution: $\mathbf{w} = (1, 0)$, $b = -1$, multipliers $1/2$, $1/4$, $1/4$. The street has width $2$. A point predicts $+1$ when $x_1 - 1 > 0$.


In [1]:
# Rebuild the line from the multipliers and apply it to two new points.
from fractions import Fraction

points = [(Fraction(2), Fraction(0)), (Fraction(0), Fraction(1)), (Fraction(0), Fraction(-1))]
labels = [Fraction(1), Fraction(-1), Fraction(-1)]
alpha = [Fraction(1, 2), Fraction(1, 4), Fraction(1, 4)]
w1 = sum(a * y * p[0] for a, y, p in zip(alpha, labels, points))
w2 = sum(a * y * p[1] for a, y, p in zip(alpha, labels, points))
b = labels[0] - (w1 * points[0][0] + w2 * points[0][1])
assert (w1, w2, b) == (1, 0, -1)

def predict(x1, x2):
    score = w1 * x1 + w2 * x2 + b
    if score > 0:
        return score, "+1"
    if score < 0:
        return score, "-1"
    return score, "on the boundary"

assert predict(Fraction(3), Fraction(2)) == (2, "+1")
assert predict(Fraction(1, 2), Fraction(4)) == (Fraction(-1, 2), "-1")
# (4, 0) is correct and is not a support vector: its margin is 3, so alpha would be 0.
assert 1 * (w1 * 4 + b) == 3
print("predict (3, 2)", predict(3, 2))
print("predict (1/2, 4)", predict(Fraction(1, 2), 4))


predict (3, 2) (Fraction(2, 1), '+1')
predict (1/2, 4) (Fraction(-1, 2), '-1')


## The same idea with slack

On the two-point line $x \in \{0, 1\}$ with labels $-1, +1$, the hard fit is $f(x) = 2x - 1$ and costs $2$ with no slack. With $C = 1$ the weight shortens to $1$, the balanced bias is $-1/2$, and each hinge is $1/2$. Soft cost: $3/2$.

Those points are still classified correctly. Slack here means "inside the street," not "wrong label."


In [2]:
# Replay the two-point soft margin. Correct class, positive hinge.
from fractions import Fraction

w, b = Fraction(1), Fraction(-1, 2)
rows = [(Fraction(0), Fraction(-1)), (Fraction(1), Fraction(1))]
for x, y in rows:
    score = w * x + b
    margin = y * score
    hinge = max(Fraction(0), 1 - margin)
    assert score * y > 0
    assert hinge == Fraction(1, 2)
    print("x", x, "score", score, "hinge", hinge)


x 0 score -1/2 hinge 1/2
x 1 score 1/2 hinge 1/2


## The same idea with a kernel

XOR is not a line in the original coordinates. In the product features it is the score $-x_1 x_2$, with each multiplier equal to $1/4$ for the matching kernel

$$
K_\psi(\mathbf{x}, \mathbf{z}) = \mathbf{x}\cdot\mathbf{z} + (x_1 x_2)(z_1 z_2)
$$

At $(2, 1)$ that score is $-2$, so the call is $-1$. The textbook kernel $(\mathbf{x}\cdot\mathbf{z})^2$ makes the same call on this pattern when its own multipliers, $1/8$, are the ones you use.


In [3]:
# Predict XOR at (2, 1) from the product kernel, then from (x·z)^2.
from fractions import Fraction

rows = [((1, 1), -1), ((1, -1), 1), ((-1, -1), -1), ((-1, 1), 1)]

def product_kernel(x, z):
    return x[0] * z[0] + x[1] * z[1] + (x[0] * x[1]) * (z[0] * z[1])

alpha = Fraction(1, 4)
product_score = sum(alpha * y * product_kernel(p, (2, 1)) for p, y in rows)
assert product_score == -2

def squared_dot(x, z):
    return (x[0] * z[0] + x[1] * z[1]) ** 2

poly_score = sum((1 / 8) * y * squared_dot(p, (2, 1)) for p, y in rows)
assert abs(poly_score - (-2)) < 1e-12
print("product-kernel score", product_score)
print("squared-dot score", poly_score)


product-kernel score -2
squared-dot score -2.0


## What this path was

1. Predict by the sign of $\mathbf{w}\cdot\mathbf{x}+b$.
2. Measure distance by dividing that score by $\|\mathbf{w}\|$.
3. Choose the shortest $\mathbf{w}$ whose functional margins are at least $1$. That is the widest street.
4. Read the street off the points with positive multipliers.
5. If a point must be allowed inside the street, charge $C$ times its hinge.
6. If a line in the original coordinates is the wrong shape, replace the dot product by a kernel that matches a feature map you can name.
7. With more than two classes, train one machine per class and keep ties visible.

The next check on any new data set is the same four-line habit used here: write the candidate $(\mathbf{w}, b)$, test every constraint by hand, recompute $\mathbf{w}$ from the multipliers, and only then ask Python whether the two routes agree.
